# AbSynth — Interactive Demo

Quick-start notebook for both AbSynth models.

| Model | Type | Capabilities |
|---|---|---|
| **AbSynth-A** | Autoregressive (GPT-2) | Generation, CDR infilling, perplexity scoring |
| **AbSynth-M** | Masked LM (RoBERTa) | Recovery, CDR Infilling, humanization, embeddings |

**Run cells top to bottom. Set your paths in Cell 1.**

## Cell 1 — Paths & setup

In [1]:
import os, sys

REPO_ROOT      = os.path.abspath(".")
MODEL_A_PATH   = os.path.join(REPO_ROOT, "absynth", "trained_models", "absynth-a")
MODEL_M_PATH   = os.path.join(REPO_ROOT, "absynth", "trained_models", "absynth-m")
TOKENIZER_PATH = os.path.join(REPO_ROOT, "absynth", "tokenizer")

sys.path.insert(0, REPO_ROOT)
from absynth import AbSynthA, AbSynthM

print(f"AbSynth-A model : {MODEL_A_PATH}")
print(f"AbSynth-M model : {MODEL_M_PATH}")

c:\Users\User\anaconda3\envs\absynth\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AbSynth-A model : c:\Users\User\OneDrive\Desktop\PyProject\github-repo\absynth\trained_models\absynth-a
AbSynth-M model : c:\Users\User\OneDrive\Desktop\PyProject\github-repo\absynth\trained_models\absynth-m


## Cell 2 — Load models

In [2]:
model_a = AbSynthA(MODEL_A_PATH, TOKENIZER_PATH)
model_m = AbSynthM(MODEL_M_PATH, TOKENIZER_PATH)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'RobertaTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'RobertaTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'RobertaTokenizer'.


AbSynth-A loaded | device=cuda | vocab=25 | params=21,874,176
AbSynth-M loaded | device=cuda | vocab=25 | params=17,745,433


---
## AbSynth-A

### Generation

In [3]:
seqs = model_a.generate(n=5, prompt="QVQ")
for i, s in enumerate(seqs, 1):
    print(f"{i:>2}  {s}  (len={len(s)})")

Generating: 100%|██████████| 5/5 [00:01<00:00,  3.06it/s]

 1  QVQVESGGGVVQPGRSLRLSCAASGFTFNSYGMHWVRQAPGKGLEWVAVIWYDGSNKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAREGYGGNDYWGQGTLVTVSS  (len=116)
 2  QVQQESGPGLVKPSETLSLTCAVSGGSISSSYWSWIRQPPGKGLEWIGYIHYSGSTNYNPSLKSRVTMSVDTSKNQFSLKLSSVTAADTAVYYCARVGGGDGYNHPPTVYYYMDVWGKGTTVTVSS  (len=126)
 3  QVQVQSGAEVKKPGASVKVSCKTSGYTFTNYDINWVRQAPGQGLEWMGWMNPNSGNTGYAQKFQGRVTMTRNTSISTAYMELSSLRSEDTAVYYCARVDYGGFYYYYMDVWGKGTTVTVSS  (len=121)
 4  QVQVESGGGVVQPGRSLRLSCAASGFTFSSYGMHWVRQAPGKGLEWVAVISYDGSNKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAKDSEPVIAVAGTSDYWGQGTLVTVSS  (len=122)
 5  QVQVQSGAEVKKPGASVRVSCKASGYTFTNHYIHWVRQAPGQGLEWMGLINPDDGSTDYARKFQGRVTMTRDTSTSTVYMDLSSLRSEDTAVYYCARERSITVIMVSAPNFDFWGQGTLVTVSS  (len=124)


### CDR Infilling

In [4]:
BASE_SEQ     = "EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLEWVSAISGSGGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDREDIVVVPAPRGYYYYYYMDVWGQGTTVTVSS"
INFILL_RANGE = (105, 125)

orig_len = INFILL_RANGE[1] - INFILL_RANGE[0]
print(f"Base ({len(BASE_SEQ)} aa):")
print(f"  {BASE_SEQ}")
print(f"  {' ' * INFILL_RANGE[0]}{'^' * orig_len}  ← infill region ({orig_len} aa)")
print(f"  length_tolerance=2  →  generated region will be {orig_len-2}–{orig_len+2} aa\n")

# length_range omitted — defaults to original region length ± length_tolerance
infilled = model_a.infill(BASE_SEQ, INFILL_RANGE, n=5)
for i, s in enumerate(infilled, 1):
    delta = len(s) - len(BASE_SEQ)
    print(f"{i:>2}  {s}  (len={len(s)}, Δ={delta:+d})")

Base (131 aa):
  EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLEWVSAISGSGGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDREDIVVVPAPRGYYYYYYMDVWGQGTTVTVSS
                                                                                                           ^^^^^^^^^^^^^^^^^^^^  ← infill region (20 aa)
  length_tolerance=2  →  generated region will be 18–22 aa



Infilling: 100%|██████████| 5/5 [00:00<00:00, 18.16it/s]

 1  EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLEWVSAISGSGGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDREDIVVVPAAIPYYYYGMDVWGQGTTTVTVSS  (len=131, Δ=+0)
 2  EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLEWVSAISGSGGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDREDIVVVPAAIGGYYYYYGMDVWGTVTVSS  (len=129, Δ=-2)
 3  EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLEWVSAISGSGGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDREDIVVVPAATRGEGYYYYGMDVWGTVTVSS  (len=130, Δ=-1)
 4  EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLEWVSAISGSGGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDREDIVVVPAAMRYYYGMDVWGQGTTVTVTVSS  (len=131, Δ=+0)
 5  EVQLVQSGGGLVQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLEWVSAISGSGGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDREDIVVVPAAMPGYYYYGMDVWGQGTVTVSS  (len=130, Δ=-1)


### Perplexity Scoring

In [5]:
ppls = model_a.perplexity(seqs)
ranked = sorted(zip(ppls, seqs))
print(f"{'Rank':>4}  {'PPL':>8}  Sequence")
print("-" * 80)
for rank, (p, s) in enumerate(ranked, 1):
    print(f"{rank:>4}  {p:>8.3f}  {s}")

Rank       PPL  Sequence
--------------------------------------------------------------------------------
   1     1.465  QVQVESGGGVVQPGRSLRLSCAASGFTFNSYGMHWVRQAPGKGLEWVAVIWYDGSNKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAREGYGGNDYWGQGTLVTVSS
   2     1.524  QVQVESGGGVVQPGRSLRLSCAASGFTFSSYGMHWVRQAPGKGLEWVAVISYDGSNKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAKDSEPVIAVAGTSDYWGQGTLVTVSS
   3     1.525  QVQVQSGAEVKKPGASVKVSCKTSGYTFTNYDINWVRQAPGQGLEWMGWMNPNSGNTGYAQKFQGRVTMTRNTSISTAYMELSSLRSEDTAVYYCARVDYGGFYYYYMDVWGKGTTVTVSS
   4     1.814  QVQQESGPGLVKPSETLSLTCAVSGGSISSSYWSWIRQPPGKGLEWIGYIHYSGSTNYNPSLKSRVTMSVDTSKNQFSLKLSSVTAADTAVYYCARVGGGDGYNHPPTVYYYMDVWGKGTTVTVSS
   5     2.036  QVQVQSGAEVKKPGASVRVSCKASGYTFTNHYIHWVRQAPGQGLEWMGLINPDDGSTDYARKFQGRVTMTRDTSTSTVYMDLSSLRSEDTAVYYCARERSITVIMVSAPNFDFWGQGTLVTVSS


---
## AbSynth-M

### Recovery (fill missing X positions)

In [6]:
MASKED_SEQ = "EVQLQQSGXELAXXGASVXMXCKAXXYXFTSXXXXXVKXRPGQGXEWIGXXNPSSGYSXYXQXXKDKATLXAXKSSXXAXXQXSSLXSEDSAVXXCSRPVXRXGXNFXYWGQGSXLXVXSA"

recovered = model_m.recover(MASKED_SEQ, n=3)
print(f"Input  : {MASKED_SEQ}\n")
for i, s in enumerate(recovered, 1):
    print(f"{i:>2}  {s}")

Recovering: 100%|██████████| 3/3 [00:00<00:00, 64.48it/s]

Input  : EVQLQQSGXELAXXGASVXMXCKAXXYXFTSXXXXXVKXRPGQGXEWIGXXNPSSGYSXYXQXXKDKATLXAXKSSXXAXXQXSSLXSEDSAVXXCSRPVXRXGXNFXYWGQGSXLXVXSA

 1  EVQLQQSGPELAKPGASVKMSCKASGYTFTSYWINWVKQRPGQGLEWIGIINPSSGYSNYNQKFKDKATLTADKSSSTAYMQLSSLTSEDSAVYYCSRPVYRGGYNFDYWGQGSTLTVVSA
 2  EVQLQQSGAELAKPGASVKMSCKASGYTFTSYWIHWVKQRPGQGLEWIGEINPSSGYSIYNQKFKDKATLTADKSSSTAYMQLSSLTSEDSAVYYCSRPVAREGYNFDYWGQGSTLTVVSA
 3  EVQLQQSGAELAKPGASVKMSCKASGYTFTSYWMSWVKQRPGQGLEWIGEINPSSGYSNYNQKFKDKATLTADKSSSTAYMQLSSLTSEDSAVYYCSRPVARAGSNFDYWGQGSTLTVVSA


### Redesign

In [7]:
BASE_SEQ_M     = "EVQLQQSGAELARPGASVKMSCKASGYTFTSYTMHWVKQRPGQGLEWIGYINPSSGYSNYNQKFKDKATLTADKSSSTAYMQLSSLTSEDSAVYYCSRPVVRLGYNFDYWGQGSTLTVSSA"
REDESIGN_RANGE = (95, 110)

orig_len = REDESIGN_RANGE[1] - REDESIGN_RANGE[0]
print(f"Base ({len(BASE_SEQ_M)} aa):")
print(f"  {BASE_SEQ_M}")
print(f"  {' ' * REDESIGN_RANGE[0]}{'^' * orig_len}  ← redesign region ({orig_len} aa)\n")

# length_range omitted — defaults to original region length ± length_tolerance
redesigned = model_m.redesign(BASE_SEQ_M, REDESIGN_RANGE, n=5)
for i, s in enumerate(redesigned, 1):
    delta = len(s) - len(BASE_SEQ_M)
    print(f"{i:>2}  {s}  (len={len(s)}, Δ={delta:+d})")

Base (121 aa):
  EVQLQQSGAELARPGASVKMSCKASGYTFTSYTMHWVKQRPGQGLEWIGYINPSSGYSNYNQKFKDKATLTADKSSSTAYMQLSSLTSEDSAVYYCSRPVVRLGYNFDYWGQGSTLTVSSA
                                                                                                 ^^^^^^^^^^^^^^^  ← redesign region (15 aa)



Redesigning: 100%|██████████| 5/5 [00:00<00:00, 86.23it/s]

 1  EVQLQQSGAELARPGASVKMSCKASGYTFTSYTMHWVKQRPGQGLEWIGYINPSSGYSNYNQKFKDKATLTADKSSSTAYMQLSSLTSEDSAVYYCARGSGLWLFEVWGQGSTLTVSSA  (len=119, Δ=-2)
 2  EVQLQQSGAELARPGASVKMSCKASGYTFTSYTMHWVKQRPGQGLEWIGYINPSSGYSNYNQKFKDKATLTADKSSSTAYMQLSSLTSEDSAVYYCARDGTTSSDYGDYWGQGSTLTVSSA  (len=121, Δ=+0)
 3  EVQLQQSGAELARPGASVKMSCKASGYTFTSYTMHWVKQRPGQGLEWIGYINPSSGYSNYNQKFKDKATLTADKSSSTAYMQLSSLTSEDSAVYYCARLGGDNDSYTLYDVWGQGSTLTVSSA  (len=123, Δ=+2)
 4  EVQLQQSGAELARPGASVKMSCKASGYTFTSYTMHWVKQRPGQGLEWIGYINPSSGYSNYNQKFKDKATLTADKSSSTAYMQLSSLTSEDSAVYYCARDGYIEYYFDYWGQGSTLTVSSA  (len=120, Δ=-1)
 5  EVQLQQSGAELARPGASVKMSCKASGYTFTSYTMHWVKQRPGQGLEWIGYINPSSGYSNYNQKFKDKATLTADKSSSTAYMQLSSLTSEDSAVYYCARGDGYGGFDHWGQGSTLTVSSA  (len=119, Δ=-2)


### Humanness Optimization

In [8]:
optimized, history = model_m.humanize(BASE_SEQ_M, top_n=5, rounds=3)

print(f"\nOriginal  : {BASE_SEQ_M}")
print(f"Optimized : {optimized}")
print(f"\nConfidence trajectory:")
for r, seq, conf in history:
    label = "(original)" if r == 1 else f"(round {r-1})"
    print(f"  Round {r}  conf={conf:.4f}  {label}")

Round 1/3 | avg confidence: 0.6974
  Masking 5 positions: [32, 0, 120, 54, 96]
Round 2/3 | avg confidence: 0.7300
  Masking 5 positions: [118, 113, 19, 57, 98]
Round 3/3 | avg confidence: 0.7538
  Masking 5 positions: [102, 11, 65, 101, 49]

Final avg confidence: 0.7800

Original  : EVQLQQSGAELARPGASVKMSCKASGYTFTSYTMHWVKQRPGQGLEWIGYINPSSGYSNYNQKFKDKATLTADKSSSTAYMQLSSLTSEDSAVYYCSRPVVRLGYNFDYWGQGSTLTVSSA
Optimized : QVQLQQSGAELVRPGASVKISCKASGYTFTSYYMHWVKQRPGQGLEWIGRINPSDGYTNYNQKFKGKATLTADKSSSTAYMQLSSLTSEDSAVYYCARDVVASGYNFDYWGQGTTLTVVSS

Confidence trajectory:
  Round 1  conf=0.6974  (original)
  Round 2  conf=0.7300  (round 1)
  Round 3  conf=0.7538  (round 2)
  Round 4  conf=0.7800  (round 3)


### Embedding Extraction

In [9]:
embeds = model_m.embed(seqs + redesigned)
print(f"Embedding shape: {embeds.shape}  (n_sequences, hidden_size)")

Embedding: 100%|██████████| 10/10 [00:00<00:00, 113.04it/s]

Embedding shape: (10, 768)  (n_sequences, hidden_size)


---
## Analysis

For deeper analysis, see the notebooks at repo root:

| Notebook | What it does |
|---|---|
| [scan_demo.ipynb](scan_demo.ipynb) | Positional perplexity scan (CDR peak visualization) |
| [plot_embeddings_demo.ipynb](plot_embeddings_demo.ipynb) | Token embedding PCA / t-SNE / cosine similarity |

Or run the CLI scripts directly:
```bash
python analysis/scan_evaluate.py paired_sequences.csv --output_dir results/scan
python analysis/plot_scan.py results/scan/losses.h5
python analysis/plot_embeddings.py --model autoregressive
python analysis/plot_embeddings.py --model masked
```